# lgbm_quantile_hpo — LGBM Quantile Regression × 2 (q=0.5, q=0.9)

**목적**: stacking pool 의 잔차 다양성 확보. 기존 base 들이 모두 MSE/Tweedie/Poisson 계열 → **잔차 패턴이 0.99+ corr 로 묶임**. quantile loss 는 cost surface 자체가 다르므로 **잔차 분포 자체가 다른 base** 가 나올 가능성 큼.

**핵심 설계**:
- LGBM objective='quantile', alpha={0.5, 0.9} 두 변형
- target = log1p(health) (학습은 log space, 추론 시 expm1 + clip≥0)
- log1p 는 monotonic 변환이라 quantile 성질 보존
- 전처리: 03b log1p preset (zit_only / bag_zit / ts_reverse 와 동일)
- KFold 5 (unit-level shuffle SEED=42)
- HPO N_TRIALS=30 + **warm start** (기존 reg_only/lgbm best HP enqueue)
- 모듈 무수정 — 노트북 안에서 직접 LGBMRegressor 호출

**기대**:
- 단일 val RMSE: 0.0058~0.0062 추정 (quantile loss 가 RMSE 와 다른 목적이라 plateau 0.0057 대보다 약할 수 있음)
- **잔차 corr_with_others < 0.99 가 핵심** — 다른 base 들과의 mean_corr 가 0.99 미만이면 stacking 가치 큼

**산출물**: `4_output/_temp/lgbm_q05/`, `lgbm_q09/` 에 oof_die/val_die/test_die/oof_unit/val_unit/test_unit + best_params + meta

## 1. 환경 + import

In [1]:
import os, sys, json, time

%run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. 설정 (HPO 30 trial + warm start)

In [2]:
EXP_ID_BASE = 'lgbm-quantile-001'
USER        = 'jh'
N_TRIALS    = 30
N_FOLDS     = 5
CLIP_Y_EXTREME    = True
TARGET_TRANSFORM  = 'log1p'
QUANTILE_ALPHAS   = [0.5, 0.9]   # median + high tail

OUT_DIR_ROOT = os.path.join(OUTPUT_DIR, '_temp')
def out_dir_for(alpha):
    tag = f'q{int(alpha*100):02d}'    # 0.5 → q05, 0.9 → q09
    return os.path.join(OUT_DIR_ROOT, f'lgbm_{tag}')

# 03b log1p preset (zit_only / bag_zit / ts_reverse 와 동일)
PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}

# warm start — 기존 reg_only/lgbm best HP
WARM_START_PATH = os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm', 'best_params.json')
with open(WARM_START_PATH, 'r', encoding='utf-8') as f:
    _bp = json.load(f)
_HP_RAW = _bp['best_params_resolved']
# search space 와 일치하는 11개 HP 만 추출 (objective/tweedie_variance_power/random_state/n_jobs/verbose/device 제외)
WARM_START_PARAMS = {
    k: _HP_RAW[k] for k in (
        'n_estimators','learning_rate','num_leaves','max_depth','min_child_samples',
        'subsample','colsample_bytree','reg_alpha','reg_lambda','min_split_gain','path_smooth',
    ) if k in _HP_RAW
}

print(f'EXP_ID_BASE     = {EXP_ID_BASE}')
print(f'QUANTILE_ALPHAS = {QUANTILE_ALPHAS}')
print(f'N_TRIALS        = {N_TRIALS}')
print(f'N_FOLDS         = {N_FOLDS}')
print(f'TARGET_TRANSFORM= {TARGET_TRANSFORM}')
print(f'WARM_START_PATH = {WARM_START_PATH}')
print(f'WARM_START 11 HP: {list(WARM_START_PARAMS)}')
for alpha in QUANTILE_ALPHAS:
    print(f'  out_dir_for({alpha}) = {out_dir_for(alpha)}')

EXP_ID_BASE     = lgbm-quantile-001
QUANTILE_ALPHAS = [0.5, 0.9]
N_TRIALS        = 30
N_FOLDS         = 5
TARGET_TRANSFORM= log1p
WARM_START_PATH = c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\reg_only\lgbm\best_params.json
WARM_START 11 HP: ['n_estimators', 'learning_rate', 'num_leaves', 'max_depth', 'min_child_samples', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda', 'min_split_gain', 'path_smooth']
  out_dir_for(0.5) = c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\lgbm_q50
  out_dir_for(0.9) = c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\lgbm_q90


## 3. 데이터 로드 + Y clip

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드] xs={xs.shape}, X feat_cols={len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  y_train: max={y_train_unit.max():.6f}, mean={y_train_unit.mean():.6f}, zero ratio={(y_train_unit==0).mean():.1%}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개

[데이터 로드] xs=(174572, 1091), X feat_cols=1087
  unit train=26,187, val=8,727, test=8,729
  y_train: max=0.097417, mean=0.002481, zero ratio=70.8%


## 4. 전처리 (03b log1p preset)

In [4]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'
y_train_die_log = np.log1p(y_train_die_broadcast)

n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)
print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  X_train_die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  y_train_die_broadcast (unit y): mean={y_train_die_broadcast.mean():.6f}')

# Outer KFold split (unit-level)
unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))
print(f'\nfold split: {N_FOLDS} folds (unit-level)')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 25%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=5.0): 156,772개 채움 → 잔여: 186,722
  2단계 (lot 평균, train 기준): 105,526개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(156,772) → lot(105,526) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.99, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.99
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 626)

클리닝 완

## 5. helper — LGBM quantile fold trainer + HPO objective

- `_hp_from_trial`: search space 11개 HP (lgbm_space 와 동일 분포, objective 분기 제거)
- `_train_lgbm_quantile`: 1 fold 학습 + 예측 (log space 학습 → expm1+clip 환원)

In [5]:
def _hp_from_trial(trial):
    """LGBM HP search space (lgbm_space 동일 분포, objective 외부 고정)."""
    return dict(
        n_estimators      = trial.suggest_int('n_estimators', 100, 3000),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 8, 512),
        max_depth         = trial.suggest_int('max_depth', 3, 14),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 400),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.1, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 30.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        min_split_gain    = trial.suggest_float('min_split_gain', 1e-9, 1.0, log=True),
        path_smooth       = trial.suggest_float('path_smooth', 0.0, 50.0),
    )


def _train_lgbm_quantile(X_tr, y_tr_log, X_others, hp, alpha):
    """1 fold 학습 + 예측. log space 학습 → expm1+clip 환원.

    Parameters
    ----------
    hp : dict
        11개 HP (objective/alpha 외부 적용).
    alpha : float
        quantile level (0.5 = median, 0.9 = high tail).
    X_others : list of np.array
        예측 대상 (val_split, val, test 등).

    Returns
    -------
    list of pred per X in X_others (die-level, original space, ≥0).
    """
    params = dict(hp)
    params['objective']    = 'quantile'
    params['alpha']        = alpha
    params['subsample_freq'] = 1   # subsample 활성화
    params['random_state'] = SEED
    params['n_jobs']       = -1
    params['verbose']      = -1
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr_log)
    return [np.clip(np.expm1(model.predict(X_o)), 0.0, None) for X_o in X_others]


def _mean_die_to_unit(pred_die, uid_die):
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    pred_sum = np.zeros(len(unique_units))
    cnt      = np.zeros(len(unique_units))
    np.add.at(pred_sum, inverse, pred_die)
    np.add.at(cnt,      inverse, 1.0)
    return pred_sum / cnt, unique_units


def _rmse_unit(pred_unit_arr, unique_units, y_unit_series):
    s = pd.Series(pred_unit_arr, index=unique_units).reindex(y_unit_series.index)
    return float(np.sqrt(np.mean((s.values - y_unit_series.values) ** 2)))


print('helpers 정의 완료')

helpers 정의 완료


## 6. HPO + Refit 통합 함수

1 quantile (alpha) 마다: HPO N_TRIALS → best HP 로 5-fold refit → die/unit 예측 누적.

In [6]:
def run_hpo_and_refit(alpha, n_trials, warm_params=None):
    tag    = f'q{int(alpha*100):02d}'
    exp_id = f'{EXP_ID_BASE}-{tag}'
    out_dir = out_dir_for(alpha)
    os.makedirs(out_dir, exist_ok=True)
    db_path = os.path.join(out_dir, f'optuna_{USER}_{exp_id}.db')

    print(f'\n{"="*72}')
    print(f'  [{tag}] LGBM quantile alpha={alpha}, exp_id={exp_id}')
    print(f'{"="*72}')

    def objective(trial):
        hp = _hp_from_trial(trial)
        oof_die_pred = np.full(n_train_die, np.nan)
        for tr_uidx, vl_uidx in FOLDS:
            tr_units = unit_ids_train_unique[tr_uidx]
            vl_units = unit_ids_train_unique[vl_uidx]
            tr_die_mask = np.isin(uid_train_die, tr_units)
            vl_die_mask = np.isin(uid_train_die, vl_units)
            preds = _train_lgbm_quantile(
                X_train_die[tr_die_mask],
                y_train_die_log[tr_die_mask],
                [X_train_die[vl_die_mask]],
                hp, alpha,
            )
            oof_die_pred[vl_die_mask] = preds[0]
        if np.isnan(oof_die_pred).any():
            raise RuntimeError('OOF coverage bug')
        oof_unit_arr, oof_unit_ids = _mean_die_to_unit(oof_die_pred, uid_train_die)
        return _rmse_unit(oof_unit_arr, oof_unit_ids, y_train_unit)

    study = optuna.create_study(
        direction='minimize',
        study_name=exp_id,
        storage=f'sqlite:///{db_path}',
        load_if_exists=False,
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    if warm_params:
        study.enqueue_trial(warm_params)
        print(f'  [warm start] enqueued {len(warm_params)} HP')

    print(f'  [HPO] N_TRIALS={n_trials}')
    t0 = time.time()
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f'  [HPO 완료] {time.time()-t0:.0f}s, best_value={study.best_value:.6f}')

    best_hp = study.best_trial.params

    # ── Refit ──
    print(f'  [Refit] best params 5-fold')
    t1 = time.time()
    oof_die_pred  = np.full(n_train_die, np.nan)
    val_die_pred  = np.zeros(n_val_die)
    test_die_pred = np.zeros(n_test_die)
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)
        preds = _train_lgbm_quantile(
            X_train_die[tr_die_mask],
            y_train_die_log[tr_die_mask],
            [X_train_die[vl_die_mask], X_val_die, X_test_die],
            best_hp, alpha,
        )
        p_vl, p_v, p_t = preds
        oof_die_pred[vl_die_mask] = p_vl
        val_die_pred  += p_v / N_FOLDS
        test_die_pred += p_t / N_FOLDS
        print(f'    fold {fold_idx+1}/{N_FOLDS} ({time.time()-t1:.0f}s)')
    assert not np.isnan(oof_die_pred).any()

    # unit 집계
    oof_unit_arr,  oof_unit_ids  = _mean_die_to_unit(oof_die_pred,  uid_train_die)
    val_unit_arr,  val_unit_ids  = _mean_die_to_unit(val_die_pred,  uid_val_die)
    test_unit_arr, test_unit_ids = _mean_die_to_unit(test_die_pred, uid_test_die)
    oof_unit_s  = pd.Series(oof_unit_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
    val_unit_s  = pd.Series(val_unit_arr,  index=val_unit_ids).reindex(y_val_unit.index)
    test_unit_s = pd.Series(test_unit_arr, index=test_unit_ids).reindex(y_test_unit.index)
    oof_rmse  = float(np.sqrt(np.mean((oof_unit_s.values  - y_train_unit.values) ** 2)))
    val_rmse  = float(np.sqrt(np.mean((val_unit_s.values  - y_val_unit.values)   ** 2)))
    test_rmse = float(np.sqrt(np.mean((test_unit_s.values - y_test_unit.values)  ** 2)))
    print(f'  [{tag}] oof={oof_rmse:.7f}  val={val_rmse:.7f}  test={test_rmse:.7f}')

    return {
        'tag': tag, 'alpha': alpha, 'exp_id': exp_id, 'out_dir': out_dir,
        'best_hp': best_hp, 'study_best': float(study.best_value),
        'oof_die_pred': oof_die_pred, 'val_die_pred': val_die_pred, 'test_die_pred': test_die_pred,
        'oof_unit_s': oof_unit_s, 'val_unit_s': val_unit_s, 'test_unit_s': test_unit_s,
        'oof_rmse': oof_rmse, 'val_rmse': val_rmse, 'test_rmse': test_rmse,
    }

print('run_hpo_and_refit 정의 완료')

run_hpo_and_refit 정의 완료


## 7. q=0.5 (median) 실행

In [7]:
res_q05 = run_hpo_and_refit(0.5, N_TRIALS, warm_params=WARM_START_PARAMS)


  [q50] LGBM quantile alpha=0.5, exp_id=lgbm-quantile-001-q50
  [warm start] enqueued 11 HP
  [HPO] N_TRIALS=30
  [HPO 완료] 7822s, best_value=0.005742
  [Refit] best params 5-fold
    fold 1/5 (61s)
    fold 2/5 (122s)
    fold 3/5 (182s)
    fold 4/5 (242s)
    fold 5/5 (304s)
  [q50] oof=0.0057416  val=0.0059149  test=0.0085745


## 8. q=0.9 (high tail) 실행

In [8]:
res_q09 = run_hpo_and_refit(0.9, N_TRIALS, warm_params=WARM_START_PARAMS)


  [q90] LGBM quantile alpha=0.9, exp_id=lgbm-quantile-001-q90
  [warm start] enqueued 11 HP
  [HPO] N_TRIALS=30
  [HPO 완료] 7433s, best_value=0.006487
  [Refit] best params 5-fold
    fold 1/5 (66s)
    fold 2/5 (129s)
    fold 3/5 (194s)
    fold 4/5 (256s)
    fold 5/5 (320s)
  [q90] oof=0.0064872  val=0.0065650  test=0.0089620


## 9. 산출물 저장 (oof_die / val_die / test_die / oof_unit / val_unit / test_unit + meta)

In [9]:
def save_outputs(res):
    out_dir = res['out_dir']

    def _build_die_df(uid_arr, die_id_arr, position_arr, pred, y_unit):
        df = pd.DataFrame({
            KEY_COL:     uid_arr,
            DIE_KEY_COL: die_id_arr,
            'position':  position_arr,
            'pred':      pred,
        })
        if y_unit is not None:
            df[TARGET_COL] = df[KEY_COL].map(y_unit)
        return df

    _build_die_df(uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
                  res['oof_die_pred'], y_train_unit).to_csv(os.path.join(out_dir, 'oof_die.csv'), index=False)
    _build_die_df(uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
                  res['val_die_pred'], y_val_unit).to_csv(os.path.join(out_dir, 'val_die.csv'), index=False)
    _build_die_df(uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
                  res['test_die_pred'], y_test_unit).to_csv(os.path.join(out_dir, 'test_die.csv'), index=False)

    def _build_unit_df(unit_pred_s, y_unit):
        return pd.DataFrame({
            KEY_COL:    unit_pred_s.index.values,
            'pred':     unit_pred_s.values,
            TARGET_COL: y_unit.reindex(unit_pred_s.index).values,
        })

    _build_unit_df(res['oof_unit_s'],  y_train_unit).to_csv(os.path.join(out_dir, 'oof_unit.csv'),  index=False)
    _build_unit_df(res['val_unit_s'],  y_val_unit ).to_csv(os.path.join(out_dir, 'val_unit.csv'),  index=False)
    _build_unit_df(res['test_unit_s'], y_test_unit).to_csv(os.path.join(out_dir, 'test_unit.csv'), index=False)

    with open(os.path.join(out_dir, 'best_params.json'), 'w', encoding='utf-8') as f:
        json.dump({
            'exp_id':      res['exp_id'],
            'tag':         res['tag'],
            'alpha':       res['alpha'],
            'best_hp':     res['best_hp'],
            'study_best':  res['study_best'],
        }, f, indent=2, ensure_ascii=False, default=str)

    meta = {
        'exp_id':            res['exp_id'],
        'tag':               res['tag'],
        'alpha':             res['alpha'],
        'model':             f'LGBM quantile (alpha={res["alpha"]}, log1p target)',
        'objective':         'quantile',
        'target_transform':  TARGET_TRANSFORM,
        'die_to_unit_agg':   'mean',
        'training_level':    'die-level broadcast',
        'n_folds':           N_FOLDS,
        'n_trials':          N_TRIALS,
        'oof_rmse':          res['oof_rmse'],
        'val_rmse':          res['val_rmse'],
        'test_rmse':         res['test_rmse'],
        'preprocess_PARAMS': PARAMS,
        'effective_pp_params': pp['effective_params'],
        'best_hp':           res['best_hp'],
        'CLIP_Y_EXTREME':    CLIP_Y_EXTREME,
        'feat_cols_clean_n': len(feat_cols_clean),
        'SEED':              int(SEED),
        'warm_start_source': WARM_START_PATH,
    }
    with open(os.path.join(out_dir, 'meta.json'), 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

    print(f'  [{res["tag"]}] saved → {out_dir}')
    for f_ in sorted(os.listdir(out_dir)):
        sz = os.path.getsize(os.path.join(out_dir, f_)) / 1024
        print(f'    {f_:35s}  {sz:>10,.1f} KB')

save_outputs(res_q05)
save_outputs(res_q09)

  [q50] saved → c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\lgbm_q50
    best_params.json                            0.5 KB
    meta.json                                   1.9 KB
    oof_die.csv                             5,510.5 KB
    oof_unit.csv                              975.9 KB
    optuna_jh_lgbm-quantile-001-q50.db        164.0 KB
    test_die.csv                            1,937.8 KB
    test_unit.csv                             325.5 KB
    val_die.csv                             1,937.6 KB
    val_unit.csv                              325.5 KB
  [q90] saved → c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\lgbm_q90
    best_params.json                            0.5 KB
    meta.json                                   1.9 KB
    oof_die.csv                             5,654.3 KB
    oof_unit.csv                              958.7 KB
    optuna_jh_lgbm-quantile-001-q90.db        164.0 KB
    test_die.csv                            1,914.4 KB
    test_unit.csv        

## 10. 잔차 corr 측정 (현재 stacking pool 과)

두 quantile base 의 OOF 잔차가 기존 stacking base 들과 얼마나 다른지 측정. **mean_corr_with_others < 0.99 면 stacking 추가 가치 큼**.

In [10]:
BASES = {
    'zit_only':                 '4_output/final/zit_only/oof_unit.csv',
    'bag_zit_combined_best':    '4_output/_temp/bag_zit_combined_best/oof_unit.csv',
    'bag_zit_combined_best_xy': '4_output/_temp/bag_zit_combined_best_xy/oof_unit.csv',
    'bag_zit_pp_hpo':           '4_output/_temp/bag_zit_pp_hpo/oof_unit.csv',
    'bag_zit_hpo':              '4_output/_temp/bag_zit_hpo/oof_unit.csv',
    'bag_zit_fixed_ge':         '4_output/_temp/bag_zit_fixed_ge/oof_unit.csv',
    'lgbm':                     '4_output/final/reg_only/lgbm/oof_unit.csv',
    'catboost':                 '4_output/final/reg_only/catboost/oof_unit.csv',
    'et':                       '4_output/final/reg_only/et/oof_unit.csv',
    'enet':                     '4_output/final/reg_only/enet/oof_unit.csv',
    'ts_reverse':               '4_output/_temp/two_stage_reverse/oof_unit.csv',
}
BASES_FULL = {**BASES,
    'lgbm_q05': os.path.join(res_q05['out_dir'], 'oof_unit.csv'),
    'lgbm_q09': os.path.join(res_q09['out_dir'], 'oof_unit.csv'),
}

abs_root = os.path.join(PROJECT_ROOT, '..')   # not used; paths are project-relative

dfs = {}
for k, p in BASES_FULL.items():
    abs_p = p if os.path.isabs(p) else os.path.join(PROJECT_ROOT, p)
    if os.path.exists(abs_p):
        dfs[k] = pd.read_csv(abs_p).sort_values(KEY_COL).reset_index(drop=True)
    else:
        print(f'  [경고] 누락: {abs_p}')

# health 통일 — clipped 사용 (max 0.097417). zit_only 만 1.0 max 라 제외하고 ref 만들기
ref_keys = dfs['bag_zit_combined_best'][KEY_COL].values
y_ref    = dfs['bag_zit_combined_best'][TARGET_COL].values

residuals = {}
for k, df in dfs.items():
    if (df[KEY_COL].values != ref_keys).any():
        df = df.set_index(KEY_COL).reindex(ref_keys).reset_index()
    residuals[k] = df['pred'].values - y_ref
res_df = pd.DataFrame(residuals)

# 1) 두 신규 base 의 잔차 corr_with_others
print('=' * 78)
print('  잔차 corr_with_others (낮을수록 stacking 다양성 ↑)')
print('=' * 78)
n_total = len(res_df.columns)
corr_mat = res_df.corr()
mean_corr = (corr_mat.sum(axis=1) - 1) / (n_total - 1)
for k in mean_corr.sort_values().index:
    flag = '★ 신규' if k in ('lgbm_q05', 'lgbm_q09') else ''
    print(f'  {k:28s}  mean_corr={mean_corr[k]:.6f}  {flag}')

# 2) 신규 base 만 자세히 — 각 기존 base 와의 잔차 corr
for new_k in ('lgbm_q05', 'lgbm_q09'):
    print('\n' + '-' * 78)
    print(f'  [{new_k}] 잔차 corr vs 각 기존 base')
    print('-' * 78)
    rows = []
    for k in res_df.columns:
        if k == new_k:
            continue
        c = float(np.corrcoef(res_df[new_k], res_df[k])[0, 1])
        rows.append((k, c))
    rows.sort(key=lambda x: x[1])
    for k, c in rows:
        flag = '★ 다양성' if c < 0.99 else ('  보통' if c < 0.997 else '↓ 거의 같음')
        print(f'    {k:28s}  res_corr={c:.6f}  {flag}')

  잔차 corr_with_others (낮을수록 stacking 다양성 ↑)
  lgbm_q09                      mean_corr=0.940800  ★ 신규
  lgbm_q05                      mean_corr=0.977615  ★ 신규
  enet                          mean_corr=0.983721  
  bag_zit_fixed_ge              mean_corr=0.986361  
  et                            mean_corr=0.986855  
  zit_only                      mean_corr=0.990411  
  catboost                      mean_corr=0.990517  
  lgbm                          mean_corr=0.990947  
  bag_zit_combined_best_xy      mean_corr=0.991000  
  bag_zit_combined_best         mean_corr=0.991067  
  bag_zit_hpo                   mean_corr=0.991083  
  ts_reverse                    mean_corr=0.991211  
  bag_zit_pp_hpo                mean_corr=0.991398  

------------------------------------------------------------------------------
  [lgbm_q05] 잔차 corr vs 각 기존 base
------------------------------------------------------------------------------
    lgbm_q09                      res_corr=0.914916  ★ 다양성
    bag

## 11. 요약

In [11]:
print('=' * 78)
print(f'  LGBM Quantile 두 base — 결과 요약')
print('=' * 78)
print(f'  EXP_ID_BASE       : {EXP_ID_BASE}')
print(f'  PP source         : 03b log1p preset')
print(f'  N_FOLDS / N_TRIALS: {N_FOLDS} / {N_TRIALS}')
print(f'  warm start        : 기존 reg_only/lgbm best HP enqueue')
print(f'  feat cols (clean) : {len(feat_cols_clean)}')
print('-' * 78)
print(f'  {"tag":10s}  {"alpha":>5s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
for r in (res_q05, res_q09):
    print(f'  {r["tag"]:10s}  {r["alpha"]:5.1f}  '
          f'{r["oof_rmse"]:11.7f}  {r["val_rmse"]:11.7f}  {r["test_rmse"]:11.7f}')
print('-' * 78)
print(f'  비교 (현재 stacking pool):')
print(f'    zit_only          val=0.005709, test=0.008414')
print(f'    bag_zit_hpo        val=0.005711, test=0.008417')
print(f'    lgbm (poisson)     val=0.005731, test=0.008429')
print(f'    et                 val=0.005758, test=0.008452')
print(f'    enet               val=0.005781, test=0.008459')
print(f'    stacking_full_ts   val=0.005701, test=0.008407 ← current best')
print('=' * 78)
print(f'\n  Stacking 통합 후보 경로:')
print(f'    {res_q05["out_dir"]}/oof_unit.csv, val_unit.csv, test_unit.csv')
print(f'    {res_q09["out_dir"]}/oof_unit.csv, val_unit.csv, test_unit.csv')
print(f'\n  → 잔차 mean_corr_with_others < 0.99 이면 stacking 재학습으로 진행')
print('=' * 78)

  LGBM Quantile 두 base — 결과 요약
  EXP_ID_BASE       : lgbm-quantile-001
  PP source         : 03b log1p preset
  N_FOLDS / N_TRIALS: 5 / 30
  warm start        : 기존 reg_only/lgbm best HP enqueue
  feat cols (clean) : 568
------------------------------------------------------------------------------
  tag         alpha          OOF          val         test
  q50           0.5    0.0057416    0.0059149    0.0085745
  q90           0.9    0.0064872    0.0065650    0.0089620
------------------------------------------------------------------------------
  비교 (현재 stacking pool):
    zit_only          val=0.005709, test=0.008414
    bag_zit_hpo        val=0.005711, test=0.008417
    lgbm (poisson)     val=0.005731, test=0.008429
    et                 val=0.005758, test=0.008452
    enet               val=0.005781, test=0.008459
    stacking_full_ts   val=0.005701, test=0.008407 ← current best

  Stacking 통합 후보 경로:
    c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\lgbm_q50/oof_unit.csv, v